## Workflow guide
This notebook reads the four curated Gold tables, creates a controlled export volume and writes a single CSV output per dashboard dataset for manual use in Power BI.

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "clinical_portfolio"
BASE = f"{CATALOG}.{SCHEMA}"

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.dashboard_exports"
)

EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/dashboard_exports"

quality_metrics = spark.table(f"{BASE}.gold_quality_metrics")
condition_summary = spark.table(f"{BASE}.gold_condition_summary")
patient_utilization = spark.table(f"{BASE}.gold_patient_utilization")
cohort_demographics = spark.table(f"{BASE}.gold_cohort_demographics")

print(f"Export folder ready: {EXPORT_PATH}")

Export folder ready: /Volumes/workspace/clinical_portfolio/dashboard_exports


In [0]:
exports = {
    "quality_metrics": quality_metrics,
    "condition_summary": condition_summary,
    "patient_utilization": patient_utilization,
    "cohort_demographics": cohort_demographics
}

for file_name, dataframe in exports.items():
    (
        dataframe
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{EXPORT_PATH}/{file_name}")
    )

display(dbutils.fs.ls(EXPORT_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/clinical_portfolio/dashboard_exports/cohort_demographics/,cohort_demographics/,0,1790274052074
dbfs:/Volumes/workspace/clinical_portfolio/dashboard_exports/condition_summary/,condition_summary/,0,1790274052074
dbfs:/Volumes/workspace/clinical_portfolio/dashboard_exports/patient_utilization/,patient_utilization/,0,1790274052074
dbfs:/Volumes/workspace/clinical_portfolio/dashboard_exports/quality_metrics/,quality_metrics/,0,1790274052075


### Verify dashboard exports
Each folder contains one CSV part file. The dashboard imports these four curated outputs: quality metrics, condition summary, patient utilization and cohort demographics.

# Power BI export

This notebook exports only the curated Gold datasets needed by the dashboard. The data is synthetic and aggregated; no credentials, private paths or real patient data are exported. `coalesce(1)` is used because this small portfolio dataset is downloaded manually into Power BI.